# SNI-21 — structured-target support audit

Audit CPU-only ini menghitung dukungan instance, gambar, dan identitas sumber untuk setiap nilai target terstruktur. Tidak ada training, inference, atau akses test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
import coffee_detector
os.chdir(REPO)
print('IMPORT:', coffee_detector.__file__)

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1/structured_target_audit/structured_target_support.json'
if not (DATA_ROOT / 'faruq_grouped_manifest.json').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        members = [m for m in archive.getmembers() if (
            m.name.endswith('/data.yaml')
            or m.name.endswith('/faruq_grouped_manifest.json')
            or '/train/labels/' in m.name
            or '/val/labels/' in m.name
        )]
        archive.extractall('/content', members=members, filter='data')
    (DATA_ROOT / 'train/images').mkdir(parents=True, exist_ok=True)
    (DATA_ROOT / 'val/images').mkdir(parents=True, exist_ok=True)
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT:', PROJECT_ROOT)
print('DATA   :', DATA_ROOT)
print('OUTPUT :', OUTPUT)

In [ ]:
from coffee_detector.analysis.sni21_structured_target_audit import audit_sni21_structured_targets

report = audit_sni21_structured_targets(DATA_ROOT, OUTPUT)
assert report['training_executed'] is False
assert report['inference_executed'] is False
assert report['test_images_accessed'] is False
print('AUDIT SELESAI:', report['decision'])

In [ ]:
import pandas as pd
from IPython.display import display

task_table = pd.DataFrame(report['task_rows'])
value_table = pd.DataFrame(report['value_rows'])
display(task_table.style.format({
    'train_observed_fraction': '{:.2%}',
    'val_observed_fraction': '{:.2%}',
}))
display(value_table.loc[~value_table['statistically_supported']])
print('STATISTICALLY READY:', report['statistically_ready'])
print('BLOCKED TASKS      :', report['blocked_tasks'])
print('EXPERT REVIEW      :', report['domain_expert_review_tasks'])
print('TRAINING           :', report['training_executed'])
print('TEST ACCESSED      :', report['test_images_accessed'])
print('SUMMARY            :', OUTPUT)
print('Kirim task table dan tabel nilai unsupported. Jangan training model baru.')